# Lab 00: Environment Setup, Multi-Domain BigQuery & Knowledge Catalog
Building Enterprise AI Agents on **Gemini Enterprise Agent Platform (GEAP)**: Pulse Telco Workshop

---

## 🎯 Architectural Context & Objectives
On an architecture diagram, connecting an AI agent to an enterprise data warehouse looks like a single arrow. In a production environment, that arrow crosses three distinct control planes: **data residency boundaries**, **semantic governance**, and **identity-bound access control**.

In this foundational lab, you provision the Google Cloud control plane and data plane that underpin all labs in the **Pulse Telco** workshop:
1. Configure project variables and European residency boundaries across three location tiers:
   * **europe-west4** (Netherlands) for regional stateful compute and orchestration services: **Agent Runtime**, **Sessions**, **Memory Bank**, **Code Execution Sandbox**, **Feedback**, **Agent Registry**, **Cloud Run**, and managed **Evaluation**.
   * **EU** multi-region for **Gemini models**, **Gemini Live voice**, **Skill Registry**, **Model Armor templates**, and **Knowledge Catalog**.
   * **EU** multi-region for **BigQuery** analytical datasets.
2. Bind the workshop to **your existing Google Cloud project**.
3. Enable all required **Gemini Enterprise Agent Platform (GEAP)**, **BigQuery**, **Knowledge Catalog**, **Agent Registry**, **Agent Gateway**, **Model Armor**, and observability APIs (approximately 30 APIs).
4. Seed **Pulse Telco's three governed BigQuery data domains** (CRM, Billing, and Network), attach custom **Knowledge Catalog Aspect Types** (*telco-governance* and *telco-semantics*), and provision two IAM User Personas (*persona-alex-exec* and *persona-elena-care*) so Labs 01 to 08 can enforce real dataset-level BigQuery IAM.
5. Install and verify the **Agents CLI** (version 1.10.0) and execute an end-to-end smoke test against **gemini-3.8-flash** on the *eu* multi-region endpoint.
6. Persist all environment variables to *workshop.env* for Labs 01 to 11 using a merge-safe writer so re-running Lab 00 never overwrites resource IDs recorded by later labs.

### Key References & Official Documentation:
* 📘 [Gemini Enterprise Agent Platform (GEAP) Documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform)
* 🛠️ [Agent Development Kit (ADK) Documentation](https://adk.dev/)
* ⚙️ [GEAP Agent Runtime Documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/runtime)
* 🗂️ [Google Cloud Knowledge Catalog (Dataplex) Documentation](https://docs.cloud.google.com/knowledge-catalog/docs/introduction)
* 📊 [BigQuery Documentation](https://docs.cloud.google.com/bigquery/docs/introduction)
* 📦 [GitHub (google/agents-cli)](https://github.com/google/agents-cli)

---

## 🗺️ Workshop Roadmap (Labs 00 to 11) & Data Architecture

```mermaid
flowchart LR
    L0["Lab 00<br/>Setup, BQ & Catalog"] --> L1["Lab 01<br/>Local Multi-Agent ADK"]
    L1 --> L2["Lab 02<br/>First Deploy & Registry"]
    L2 --> L3["Lab 03<br/>Remote MCP & Skills"]
    L3 --> L4["Lab 04<br/>Sessions, Memory & Sandbox"]
    L4 --> L5["Lab 05<br/>Model Armor Guardrails"]
    L5 --> L6["Lab 06<br/>Remote A2A & Final Deploy"]
    L6 --> L7["Lab 07<br/>Continuous Evaluation"]
    L7 --> L8["Lab 08<br/>Graph Workflows vs. n8n"]
    L8 --> L9["Lab 09<br/>Agent Gateway (Self-Study)"]
    L9 --> L10["Lab 10<br/>Gemini Enterprise (Self-Study)"]
    L10 --> L11["Lab 11<br/>Clean Teardown"]
```

```mermaid
graph TB
    subgraph EU_BQ["BigQuery Multi-Region (EU)"]
        CRM["📊 telco_crm<br/>• subscribers (60 rows)<br/>• subscriptions (75 rows)"]
        BIL["💳 telco_billing<br/>• invoices (120 rows)<br/>• payment_transactions (120 rows)"]
        NET["📡 telco_network<br/>• cell_towers (12 rows)<br/>• network_incidents (8 rows)"]
    end

    subgraph EU_KC["Knowledge Catalog (eu)"]
        GOV["🛡️ AspectType: telco-governance<br/>• domain_owner & data_steward<br/>• pii_classification<br/>• required_access_tier"]
        SEM["📐 AspectType: telco-semantics<br/>• primary_key & join_keys<br/>• canonical_kpi_formula<br/>• business_rules"]
    end

    GOV -->|Attached to @bigquery Entries| CRM
    GOV -->|Attached to @bigquery Entries| BIL
    GOV -->|Attached to @bigquery Entries| NET
    SEM -->|Attached to @bigquery Entries| CRM
    SEM -->|Attached to @bigquery Entries| BIL
    SEM -->|Attached to @bigquery Entries| NET
```

> [!NOTE]
> **2026 Product Naming & Fast Inner-Loop Development Flow:**
> * **Gemini platform (formerly Gemini Enterprise Agent Platform / GEAP)**: Since October 2026 ("Gemini is the single brand"), the external platform name is **Gemini platform**, while documentation URLs and underlying CLI/API resource identifiers remain unchanged.
> * **Knowledge Catalog**: The current product name for Dataplex Universal Catalog (*dataplex.googleapis.com*).
> * **Two Cloud Deployments Across the Core Track**: Rebuilding a container image for every prompt or tool change slows engineering velocity to a crawl. Instead, you deploy *pulse-concierge* v1 once in **Lab 02** to provision the managed Agent Runtime instance and its cryptographic SPIFFE Agent Identity. In **Labs 03, 04, and 05**, you iterate locally in-process against that deployed instance's cloud-managed **Sessions**, **Memory Bank**, **Code Execution Sandbox**, **MCP servers**, and **Model Armor templates**, giving you a sub-second feedback loop without waiting 6 minutes per container build. You then perform one final cloud deployment in **Lab 06**.

## 📖 Glossary (Read Once, Referenced Across All Labs)

If you are coming from **Microsoft Azure / Entra ID / OpenAI Foundry** or **n8n**, use this architectural mapping table to translate between platforms:

| Term in This Workshop | What It Actually Is | Azure / OpenAI / n8n Analogy |
| :--- | :--- | :--- |
| **Agent Runtime** (reasoningEngines) | Managed serverless container runtime for agents (formerly Vertex AI Agent Engine). One underlying resource exposed under three names: product name **Agent Runtime**, REST resource *reasoningEngines*, and Python SDK *client.agent_engines* (note that vertexai.Client is the legacy alias of agentplatform.Client; its FutureWarning is benign). | Azure Container Apps + Foundry Agent Service host |
| **europe-west4 vs eu vs EU** | Three distinct location identifiers:<br/>• **europe-west4** (single region in the Netherlands): Agent Runtime, Sessions, Memory Bank, Sandbox, Agent Registry, Cloud Run.<br/>• **eu** (EU multi-region endpoint, such as aiplatform.eu.rep.googleapis.com): Gemini 3.8 Flash, Gemini 3.8 Live, Skill Registry, Model Armor, and Knowledge Catalog.<br/>• **EU** (uppercase): BigQuery multi-region dataset location. | Regional resource group (westeurope) vs EU Data Boundary multi-region endpoint |
| **principal://agents.global.org-…** | **Agent Identity**: a system-attested workload identity built on **SPIFFE** (Google automatically provisions and rotates an X.509 certificate for each Agent Runtime instance). Lives in your organization's trust domain (*agents.global.org-<ORG_ID>.system.id.goog*). | Azure Managed Identity / Microsoft Entra Agent ID |
| **ya29.…** | A short-lived (1-hour) Google OAuth 2.0 Bearer access token. Minting one on behalf of a service account requires *roles/iam.serviceAccountTokenCreator* on that service account ([service-account impersonation](https://docs.cloud.google.com/iam/docs/service-account-impersonation)). | OAuth 2.0 Bearer access token (eyJ...) |
| **ADC vs gcloud auth login** | CLI commands authenticate with the gcloud user login store. Python SDKs (GenAI, BigQuery, Vertex AI, and ADK) authenticate with [Application Default Credentials (ADC)](https://docs.cloud.google.com/docs/authentication/application-default-credentials). When you run an agent **locally**, the agent's own identity is your ADC user. | az login vs DefaultAzureCredential |
| **AgentTool vs sub_agents** | In ADK, *AgentTool(agent=...)* implements **call-and-return**: the parent orchestrator invokes a specialist sub-agent like a typed function, receives its structured answer, and retains control of the turn so it can synthesize CRM and Billing results. Passing *sub_agents=[...]* implements **hand-off / transfer**, where control transfers to the child agent. | Foundry Connected Agents (call-and-return) vs OpenAI Agents SDK handoffs |
| **Knowledge Catalog** (dataplex.googleapis.com) | Google Cloud's unified metadata and governance catalog (formerly Dataplex Universal Catalog; API names unchanged). An **Aspect** is typed governance or business metadata attached to a table entry. | Microsoft Purview Data Map classifications & glossary terms |
| **IAP in Agent Gateway (Lab 09)** | In Lab 09, *iap.googleapis.com* acts as the **policy decision point (PDP)** evaluating CEL access policies (such as *read_only_hint* and caller principal) for Agent Gateway, rather than acting as a browser login proxy. | Azure API Management / Entra Conditional Access policy engine |

**Where to run these notebooks:** Run in any POSIX environment (Linux, macOS, WSL2 on Windows, or **Google Cloud Shell**) with Python 3.11 to 3.13, gcloud, and uv installed.

## Step 0.1: Verify gcloud CLI & Application Default Credentials (ADC)

Google Cloud separates operator CLI credentials from application SDK credentials. When you invoke gcloud in a terminal, it authenticates using the credential store populated by the standard CLI login. When Python SDKs or *agents-cli* execute inside a notebook, they read **Application Default Credentials (ADC)**.

Both credential stores must be authenticated to the **same user account** before starting; otherwise, CLI setup commands and Python SDK calls act as different IAM principals:

```bash
gcloud auth login
gcloud auth application-default login
```

Run the cell below to verify that both credential stores match and to define the shell helper that echoes each command before executing it.

In [ ]:
# Step 0.1: Shell helper and verification that gcloud CLI and Application Default Credentials use the same account
import os, subprocess, json, pathlib, re, datetime, glob, shutil
os.environ["PATH"] = f"{pathlib.Path.home() / '.local/bin'}:{os.environ.get('PATH', '')}"
os.environ["CLOUDSDK_CORE_DISABLE_PROMPTS"] = "1"   # never let gcloud wait for a (y/N) answer inside a notebook
# Why GOOGLE_API_USE_CLIENT_CERTIFICATE=false: on client-cert-enrolled machines, google-genai / vertexai otherwise
# attempts *.rep.mtls.googleapis.com, which fails DNS or crashes the aiohttp connector (harmless outside corp machines).
os.environ.setdefault("GOOGLE_API_USE_CLIENT_CERTIFICATE", "false")

# In Cloud Shell, terminal commands write ADC and active config to /tmp/tmp.*/ rather than ~/.config/gcloud.
# Sync the newest /tmp/tmp.*/ credentials into ~/.config/gcloud so the Cloud Shell Editor .venv kernel sees them automatically.
_gcloud_dir = pathlib.Path(os.environ.get("CLOUDSDK_CONFIG") or (pathlib.Path.home() / ".config" / "gcloud"))
_gcloud_dir.mkdir(parents=True, exist_ok=True)
for _tmp_adc in sorted(glob.glob("/tmp/tmp*/application_default_credentials.json"), key=os.path.getmtime, reverse=True):
    _dst_adc = _gcloud_dir / "application_default_credentials.json"
    if not _dst_adc.exists() or os.path.getmtime(_tmp_adc) > os.path.getmtime(_dst_adc):
        shutil.copy2(_tmp_adc, _dst_adc)
    break
if (_gcloud_dir / "application_default_credentials.json").exists():
    os.environ.setdefault("GOOGLE_APPLICATION_CREDENTIALS", str(_gcloud_dir / "application_default_credentials.json"))

def sh(cmd, check=True, quiet=False):
    """Run a shell command, echo it and its output, return stdout; raise on failure unless check=False."""
    if not quiet:
        print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout and not quiet:
        print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip())
        if check:
            raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r.stdout.strip()

# Check the CLI login: this account will configure and access the workshop project.
ACCOUNT = sh("gcloud config get-value account 2>/dev/null", quiet=True, check=False)
if not ACCOUNT or ACCOUNT == "(unset)":
    ACCOUNT = sh("gcloud auth list --filter=status:ACTIVE --format='value(account)' --limit=1", quiet=True, check=False)
ACCOUNT_MEMBER = f"serviceAccount:{ACCOUNT}" if ACCOUNT.endswith(".gserviceaccount.com") else f"user:{ACCOUNT}"
print("gcloud account:", ACCOUNT, f"({ACCOUNT_MEMBER})")

# Check Application Default Credentials: the Python SDKs used in the labs authenticate with these, not with the CLI login.
sh("gcloud auth application-default print-access-token >/dev/null && echo 'ADC: OK'")

# Both logins must be the SAME account, otherwise gcloud and the Python SDKs act as different identities.
import urllib.request
adc_token = sh("gcloud auth application-default print-access-token", quiet=True)
adc_email = json.load(urllib.request.urlopen(f"https://oauth2.googleapis.com/tokeninfo?access_token={adc_token}")).get("email", "?")
print("ADC account:   ", adc_email)
if adc_email != ACCOUNT:
    print("WARNING: gcloud and ADC use different accounts - re-run: gcloud auth application-default login")


## Step 0.2: Configure Workshop Names & European Locations

The workshop runs inside **an existing Google Cloud project that you supply** (such as *pulse-telco-ws-NNNN*). Leave *PROJECT_ID* empty to read the active project from your gcloud configuration, or set the project ID string explicitly.

Architecturally, European deployments on the Gemini platform split locations by workload behavior:
* **Stateful regional runtime plane (europe-west4, Netherlands)**: Hosts container compute, conversational session state, long-term memory, code execution sandboxes, and registries close to European users.
* **Stateless EU multi-region control and inference plane (eu / EU)**: Routes Gemini text and live voice inference, Model Armor guardrails, Knowledge Catalog metadata, and BigQuery analytical storage across European Union data centers without leaving the EU data boundary.

No billing-account or project-creation permissions are required because billing is already attached to the project. You only need project-level administrative roles (Owner, or Editor combined with Project IAM Admin and Service Account Admin).

| Variable | Default Value | Purpose |
| :--- | :--- | :--- |
| PROJECT_ID | "" (reads active gcloud project) | Your existing Google Cloud project (for example, pulse-telco-ws-NNNN) |
| REGION | europe-west4 (Netherlands) | Regional plane for Agent Runtime, Sessions, Memory Bank, Sandbox, Agent Registry, Cloud Run & Evaluation |
| MODEL_LOCATION | eu | European multi-region endpoint serving gemini-3.8-flash |
| LIVE_LOCATION | eu | European multi-region endpoint serving gemini-3.8-live (bidirectional voice) |
| BQ_LOCATION | EU | BigQuery European multi-region for the CRM, Billing, and Network datasets |
| PULSE_SKILLS_LOCATION | eu | Agent Registry Skills multi-region endpoint (locations/eu/skills) |
| MA_LOCATION | eu | Model Armor templates multi-region endpoint (modelarmor.eu.rep.googleapis.com) |
| MODEL | gemini-3.8-flash | Primary Gemini model used by all Pulse Telco agents |
| AGENT_NAME | pulse-concierge | Root ADK agent directory and Agent Runtime instance name |

In [ ]:
# Step 0.2: Workshop settings (PROJECT_ID is the only value you normally need to set)
PROJECT_ID            = ""                  # your project, e.g. "pulse-telco-ws-4351"; empty = gcloud config
REGION                = "europe-west4"      # Agent Runtime, Sessions, Memory Bank, Sandbox, Agent Registry, Cloud Run
MODEL_LOCATION        = "eu"                # EU multi-region endpoint that serves gemini-3.8-flash
LIVE_LOCATION         = "eu"                # EU multi-region endpoint that serves gemini-3.8-live
BQ_LOCATION           = "EU"                # BigQuery EU multi-region
PULSE_SKILLS_LOCATION = "eu"                # Agent Registry Skills location
MA_LOCATION           = "eu"                # Model Armor templates location
MODEL                 = "gemini-3.8-flash"
AGENT_NAME            = "pulse-concierge"    # name of the agent project and of the deployed agent
BQ_DATASET_CRM        = "telco_crm"
BQ_DATASET_BILLING    = "telco_billing"
BQ_DATASET_NETWORK    = "telco_network"

## Step 0.3: Verify Access to Your Project

Google Cloud identifies every project in two ways: the human-readable **project ID** (used in BigQuery table names and REST URLs) and the immutable numeric **project number** (used when constructing Google-managed service agents and SPIFFE Agent Identity trust paths).

The cell below resolves your project ID, fetches the numeric project number, and lists the IAM roles your account holds on the project. If any subsequent step fails with "PERMISSION_DENIED", inspect this role list first.

Equivalent terminal commands:
```bash
gcloud projects describe "$PROJECT_ID" --format="value(projectNumber)"
gcloud projects get-iam-policy "$PROJECT_ID" --flatten="bindings[].members" \
  --filter="bindings.members:$(gcloud config get-value account)" --format="value(bindings.role)"
```

In [ ]:
# Step 0.3: Resolve the project, read its numeric project number, and list active IAM roles
if not PROJECT_ID:
    for _cand in [
        os.environ.get("GOOGLE_CLOUD_PROJECT", ""),
        os.environ.get("CLOUDSDK_CORE_PROJECT", ""),
        os.environ.get("DEVSHELL_PROJECT_ID", ""),
        sh("gcloud config get-value project 2>/dev/null", quiet=True, check=False),
    ]:
        if _cand and _cand != "(unset)":
            PROJECT_ID = _cand.strip()
            break
if not PROJECT_ID:
    for _cfg in sorted(glob.glob("/tmp/tmp*/configurations/config_*"), key=os.path.getmtime, reverse=True):
        _m = re.search(r"^project\s*=\s*(\S+)", pathlib.Path(_cfg).read_text(), re.MULTILINE)
        if _m:
            PROJECT_ID = _m.group(1).strip()
            break
if not PROJECT_ID:
    PROJECT_ID = sh("gcloud projects list --filter='NOT projectId:sys-*' --format='value(projectId)' --limit=1", quiet=True, check=False)
assert PROJECT_ID, "Set PROJECT_ID in Step 0.2 above, or run: gcloud config set project <YOUR_PROJECT_ID>"

PROJECT_NUMBER = sh(f"gcloud projects describe {PROJECT_ID} --format='value(projectNumber)'", quiet=True)
assert PROJECT_NUMBER, f"Project {PROJECT_ID} not found or not visible to {ACCOUNT}"
print("Project:       ", PROJECT_ID, "(number", PROJECT_NUMBER + ")")

roles = sh(f"gcloud projects get-iam-policy {PROJECT_ID} --flatten='bindings[].members' "
           f"--filter='bindings.members:{ACCOUNT_MEMBER}' --format='value(bindings.role)'", quiet=True, check=False)
print("Your roles:    ", ", ".join(sorted(roles.split())) or "(could not read the IAM policy)")


## Step 0.4: Isolate gcloud in a Named Configuration (geap-telco-workshop)

To keep this workshop from modifying your personal default gcloud profile, we create and activate a dedicated configuration named *geap-telco-workshop* and set the workshop project as the quota project on your Application Default Credentials (ADC). Setting the ADC quota project ensures that API requests made by Python SDKs bill quota and usage against your workshop project.

Equivalent terminal commands:
```bash
gcloud config configurations create geap-telco-workshop || gcloud config configurations activate geap-telco-workshop
gcloud config set project "$PROJECT_ID"
gcloud config set compute/region "$REGION"
gcloud auth application-default set-quota-project "$PROJECT_ID"
```

In [ ]:
# Step 0.4: Point gcloud at the workshop project through its own configuration "geap-telco-workshop"
sh("gcloud config configurations describe geap-telco-workshop >/dev/null 2>&1 || gcloud config configurations create geap-telco-workshop", check=False)
sh("gcloud config configurations activate geap-telco-workshop")

sh(f"gcloud config set account {ACCOUNT}")
sh(f"gcloud config set project {PROJECT_ID}")
sh(f"gcloud config set compute/region {REGION}")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

## Step 0.5: Enable Required Google Cloud APIs (~30 APIs)

Enabling a Google Cloud API does two things in the engine room: it activates the service's regional REST and gRPC endpoints for your project and registers its internal service-agent identity hooks. We enable all ~30 APIs required across Labs 00 to 08 in batches of 20 now so that **BigQuery**, **Knowledge Catalog** (*dataplex.googleapis.com*), **Agent Runtime** (*aiplatform.googleapis.com*), and **IAM Credentials** (*iamcredentials.googleapis.com*) are warm before we provision data or agents.

| Workshop Lab | Google Cloud APIs Enabled |
| :--- | :--- |
| **Foundation (All Labs)** | serviceusage, cloudresourcemanager, iam, iamcredentials, orgpolicy |
| **Lab 00 (Data & Catalog)** | bigquery.googleapis.com, dataplex.googleapis.com (**Knowledge Catalog**) |
| **Labs 01 to 02 (Build & Deploy)** | aiplatform.googleapis.com (**GEAP Agent Runtime**), storage, cloudbuild, artifactregistry, agentregistry.googleapis.com, discoveryengine.googleapis.com |
| **Labs 03 & 06 (MCP & A2A)** | run.googleapis.com (**Cloud Run** for custom IAM MCP & NOC A2A agent), compute.googleapis.com |
| **Lab 04 (Observability)** | logging, monitoring, cloudtrace, telemetry, observability, apphub, apptopology, cloudapiregistry |
| **Lab 05 (Guardrails)** | modelarmor.googleapis.com (**Model Armor**), dlp.googleapis.com (**Sensitive Data Protection**) |
| **Lab 09 (Agent Gateway Architecture)** | networkservices.googleapis.com, networksecurity.googleapis.com (**Agent Gateway**), dns, iap.googleapis.com, secretmanager |

Equivalent terminal command:
```bash
gcloud services enable aiplatform.googleapis.com bigquery.googleapis.com dataplex.googleapis.com \
  agentregistry.googleapis.com modelarmor.googleapis.com networkservices.googleapis.com \
  networksecurity.googleapis.com iap.googleapis.com run.googleapis.com cloudbuild.googleapis.com \
  artifactregistry.googleapis.com discoveryengine.googleapis.com logging.googleapis.com \
  cloudtrace.googleapis.com monitoring.googleapis.com --project="$PROJECT_ID"
```

In [ ]:
# Step 0.5: Enable every API the workshop needs in batches of 20
APIS = [
    # platform basics
    "serviceusage.googleapis.com", "cloudresourcemanager.googleapis.com", "iam.googleapis.com",
    "iamcredentials.googleapis.com", "orgpolicy.googleapis.com",
    # Lab00: multi-domain BigQuery + Knowledge Catalog (Dataplex)
    "bigquery.googleapis.com", "dataplex.googleapis.com",
    # Lab01-02: Agent Platform, builds, registry, Gemini Enterprise
    "aiplatform.googleapis.com", "storage.googleapis.com", "cloudbuild.googleapis.com",
    "artifactregistry.googleapis.com", "agentregistry.googleapis.com", "discoveryengine.googleapis.com",
    # Lab03 & Lab06: enterprise MCP and A2A services on Cloud Run
    "run.googleapis.com", "compute.googleapis.com",
    # Lab04: observability
    "logging.googleapis.com", "monitoring.googleapis.com", "cloudtrace.googleapis.com",
    "telemetry.googleapis.com", "observability.googleapis.com", "apphub.googleapis.com",
    "apptopology.googleapis.com", "cloudapiregistry.googleapis.com",
    # Lab05: Model Armor
    "modelarmor.googleapis.com", "dlp.googleapis.com",
    # Lab09: gateway, identity, policies
    "networkservices.googleapis.com", "networksecurity.googleapis.com", "dns.googleapis.com",
    "iap.googleapis.com", "secretmanager.googleapis.com",
]

for i in range(0, len(APIS), 20):
    sh(f"gcloud services enable {' '.join(APIS[i:i+20])} --project={PROJECT_ID}")

enabled = set(sh(f"gcloud services list --enabled --project={PROJECT_ID} --format='value(config.name)'", quiet=True).split())
missing = [a for a in APIS if a not in enabled]
print("missing:", missing or "none - all APIs enabled")

sh(f"gcloud auth application-default set-quota-project {PROJECT_ID} --quiet", check=False)

## Step 0.6: Seed Pulse Telco's Three BigQuery Domains (EU)

**Pulse Telco** operates across five Central European markets (PRG Prague, BRN Brno, OVA Ostrava, BTS Bratislava, and VIE Vienna). Rather than dumping all operational records into a single flat dataset, Pulse Telco enforces domain-driven data boundaries across **three governed BigQuery datasets** in the EU multi-region. Separating these domains at the dataset level allows BigQuery IAM to grant customer-care specialists access to *telco_crm* while denying access to financial invoices in *telco_billing* and infrastructure telemetry in *telco_network*.

| BigQuery Dataset | Tables | Key Columns & Business Purpose |
| :--- | :--- | :--- |
| **telco_crm** | subscribers (60 rows)<br/>subscriptions (75 rows) | subscriber_id, customer_segment (consumer, sme, enterprise), region_code, ML churn_risk_score (0.0 to 1.0), nps_band, primary_tower_id, plan_type, monthly_recurring_eur |
| **telco_billing** | invoices (120 rows)<br/>payment_transactions (120 rows) | invoice_id, subscriber_id, billing_period (2026-07, 2026-08), total_amount_eur, payment_status (paid, overdue, disputed), days_past_due, payment_note |
| **telco_network** | cell_towers (12 rows)<br/>network_incidents (8 rows) | tower_id (TWR-PRG-02, TWR-BRN-02, ...), radio_type (5G-SA, 5G-NSA, LTE-A), avg_utilization_pct, packet_loss_pct, operational_status, incident_id (INC-7001, INC-7002) |

The cell below creates the three EU datasets, loads the JSON seed records from the local data directory, and stamps each table with governance descriptions and labels from *data/knowledge_catalog_specs.json* so both BigQuery schema inspection and Knowledge Catalog surface identical metadata.

In [ ]:
# Step 0.6: Seed the 3 BigQuery datasets (EU multi-region) and attach table descriptions and labels
import time, requests, google.auth
from google.auth.transport.requests import Request
from google.cloud import bigquery

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "labs" else pathlib.Path.cwd()
DATA_DIR  = REPO_ROOT / "data"
KC_SPECS  = json.loads((DATA_DIR / "knowledge_catalog_specs.json").read_text())

bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

DATASETS = {
    BQ_DATASET_CRM:     "Pulse Telco CRM domain: subscribers, segments, ML churn risk scores, and active mobile/fiber tariff subscriptions.",
    BQ_DATASET_BILLING: "Pulse Telco Billing & Revenue Assurance domain: monthly EUR invoices, delinquency, payment transactions, and dispute notes.",
    BQ_DATASET_NETWORK: "Pulse Telco Network Operations (NOC) domain: 5G/LTE cell towers, utilization, packet loss, and outage incidents.",
}

for ds_id, desc in DATASETS.items():
    ds = bigquery.Dataset(f"{PROJECT_ID}.{ds_id}")
    ds.location = BQ_LOCATION
    ds.description = desc
    bq.create_dataset(ds, exists_ok=True)
    print(f"BigQuery dataset ready: {PROJECT_ID}.{ds_id} ({BQ_LOCATION})")

TABLES = [
    (BQ_DATASET_CRM,     "subscribers",          "crm_subscribers.json"),
    (BQ_DATASET_CRM,     "subscriptions",        "crm_subscriptions.json"),
    (BQ_DATASET_BILLING, "invoices",             "billing_invoices.json"),
    (BQ_DATASET_BILLING, "payment_transactions", "billing_transactions.json"),
    (BQ_DATASET_NETWORK, "cell_towers",          "network_cell_towers.json"),
    (BQ_DATASET_NETWORK, "network_incidents",    "network_incidents.json"),
]

for ds_id, tbl_id, filename in TABLES:
    fqn = f"{PROJECT_ID}.{ds_id}.{tbl_id}"
    rows = json.loads((DATA_DIR / filename).read_text())
    bq.load_table_from_json(
        rows, fqn, job_config=bigquery.LoadJobConfig(autodetect=True, write_disposition="WRITE_TRUNCATE")
    ).result()
    # Attach rich description + governance labels directly on the BigQuery table so both BigQuery MCP and Knowledge Catalog MCP surface them.
    t_meta = KC_SPECS["table_metadata"][f"{ds_id}.{tbl_id}"]
    tbl = bq.get_table(fqn)
    tbl.description = (
        f"{t_meta['description']} | "
        f"PK: {t_meta['semantics']['primary_key']} | "
        f"Joins: {t_meta['semantics']['join_keys']} | "
        f"KPI Formula: {t_meta['semantics']['canonical_kpi_formula']} | "
        f"Rule: {t_meta['semantics']['business_rules']}"
    )
    tbl.labels = {
        "data_domain": t_meta["governance"]["data_domain"].lower(),
        "pii_tier": t_meta["governance"]["pii_classification"].lower(),
        "access_tier": t_meta["governance"]["required_access_tier"].lower(),
    }
    bq.update_table(tbl, ["description", "labels"])
    print(f"  loaded & described {fqn:<42} ({len(rows)} rows)")

### Step 0.6b: Create Knowledge Catalog Aspect Types (eu) & Attach Entry Aspects

When an LLM generates SQL or summarizes KPIs from bare column names alone, semantic drift is inevitable: one prompt treats any overdue invoice as "delinquent," while another includes disputed bills or suspended subscriptions. To eliminate guesswork, we configure **Knowledge Catalog** (*dataplex.googleapis.com*) in the *eu* location with two custom **Aspect Types** and attach them to the auto-synchronized BigQuery table entries:
1. **telco-governance**: Records the domain owner, data steward, PII classification (NONE, PSEUDONYMIZED_CUSTOMER, FINANCIAL_SENSITIVE), required token access tier, and EU residency requirement.
2. **telco-semantics**: Defines the primary key, cross-domain join keys, canonical KPI SQL formulas (High Churn Risk Rate, Active Subscription ARPU, Delinquent Exposure, Congested Tower Ratio), and mandatory business filtering rules.

Both the **Orchestrator Agent** and the **Domain Expert Sub-Agents** consult Knowledge Catalog (locally via snapshot in Lab 01, and live via *https://dataplex.googleapis.com/mcp* in Labs 03 to 08) before answering analytical questions.

You can inspect these Aspect Types and Entries from the terminal at any time:
```bash
gcloud dataplex aspect-types list --project="$PROJECT_ID" --location=eu
gcloud dataplex entries lookup --project="$PROJECT_ID" --location=eu \
  --entry="projects/$PROJECT_ID/locations/eu/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/$PROJECT_ID/datasets/telco_crm/tables/subscribers"
```

In [ ]:
# Step 0.6b: Create Knowledge Catalog (Dataplex) Aspect Types in 'eu' and attach Aspects to the BigQuery Table Entries
def gcp_token():
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    creds.refresh(Request())
    return creds.token

DP_API = "https://dataplex.googleapis.com/v1"
H = {"Authorization": f"Bearer {gcp_token()}", "Content-Type": "application/json", "X-Goog-User-Project": PROJECT_ID}

for aspect_id, spec in KC_SPECS["aspect_types"].items():
    aspect_url = f"{DP_API}/projects/{PROJECT_ID}/locations/eu/aspectTypes/{aspect_id}"
    if requests.get(aspect_url, headers=H).status_code == 200:
        print(f"Knowledge Catalog AspectType exists : {aspect_id}")
        continue
    record_fields = []
    for idx, f in enumerate(spec["fields"], start=1):
        record_fields.append({
            "name": f["name"],
            "type": "BOOL" if f["type"] == "bool" else "STRING",
            "index": idx,
            "annotations": {"description": f["description"]},
            "constraints": {"required": False},
        })
    body = {
        "displayName": spec["display_name"],
        "description": spec["description"],
        "metadataTemplate": {
            "name": aspect_id.replace("-", "_"),
            "type": "RECORD",
            "recordFields": record_fields,
        },
    }
    r = requests.post(f"{DP_API}/projects/{PROJECT_ID}/locations/eu/aspectTypes", params={"aspectTypeId": aspect_id}, headers=H, json=body)
    if r.status_code in (200, 201):
        op = r.json()
        if "name" in op and "/operations/" in op["name"]:
            for _ in range(15):
                if requests.get(f"{DP_API}/{op['name']}", headers=H).json().get("done"):
                    break
                time.sleep(2)
        print(f"Knowledge Catalog AspectType created: {aspect_id}")
    else:
        print(f"AspectType {aspect_id} response ({r.status_code}): {r.text[:200]}")

# Attach telco-governance and telco-semantics aspects to each BigQuery Entry in Knowledge Catalog (location 'eu').
# BigQuery to Knowledge Catalog automatic entry sync takes ~10-20s on a brand-new dataset, so we retry briefly if 404.
for ds_id, tbl_id, _ in TABLES:
    entry_name = f"projects/{PROJECT_ID}/locations/eu/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{ds_id}/tables/{tbl_id}"
    t_meta = KC_SPECS["table_metadata"][f"{ds_id}.{tbl_id}"]
    gov_key = f"{PROJECT_NUMBER}.eu.telco-governance"
    sem_key = f"{PROJECT_NUMBER}.eu.telco-semantics"
    payload = {
        "aspects": {
            gov_key: {
                "aspectType": f"projects/{PROJECT_NUMBER}/locations/eu/aspectTypes/telco-governance",
                "data": t_meta["governance"],
            },
            sem_key: {
                "aspectType": f"projects/{PROJECT_NUMBER}/locations/eu/aspectTypes/telco-semantics",
                "data": t_meta["semantics"],
            },
        }
    }
    for attempt in range(8):
        r = requests.patch(f"{DP_API}/{entry_name}", params={"updateMask": "aspects"}, headers=H, json=payload)
        if r.status_code == 200:
            print(f"  Attached KC aspects (telco-governance, telco-semantics) -> {ds_id}.{tbl_id}")
            break
        time.sleep(4)
    else:
        print(f"  Note: KC entry sync still in progress for {ds_id}.{tbl_id} ({r.status_code}); table description & labels are already live.")

### Step 0.6c: Pre-Provision the Two Workshop IAM User Personas (Admin & Specialist) & Dataset-Level BigQuery IAM

Prompt instructions ("do not show billing records to customer-care agents") are not a security boundary. In a real enterprise architecture, access control must be enforced by the underlying data store against the authenticated identity of the user asking the question.

To demonstrate **OAuth 2.0 User Token Delegation**, **Dataset-Level BigQuery IAM Enforcement**, and **Dual Attribution (User + Agent Identity)** in Cloud Audit Logs (*cloudaudit.googleapis.com/data_access*), we provision **two IAM User Personas** with distinct dataset entitlements:

| IAM User Persona | Role | GCP IAM Principal | Project Role | BigQuery Dataset ACL (roles/bigquery.dataViewer) | Blocked Datasets (BigQuery IAM 403) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **persona-alex-exec** | **Alex Novak** (Admin / VP of Commercial Analytics) | persona-alex-exec@&lt;PROJECT_ID&gt;.iam.gserviceaccount.com | roles/bigquery.jobUser | telco_crm, telco_billing, telco_network | None (Full Access) |
| **persona-elena-care** | **Elena Kovac** (Specialist / Customer Care Lead) | persona-elena-care@&lt;PROJECT_ID&gt;.iam.gserviceaccount.com | roles/bigquery.jobUser | **telco_crm ONLY** | **telco_billing, telco_network** |

> [!NOTE]
> **How This Maps to Production (Microsoft 365 / Entra ID + Cloud Identity vs. Workshop Sandbox)**
> * **Production Enterprise Environment (Entra ID + Cloud Identity SAML/OIDC)**: Human employees and security groups are synchronized from **Microsoft Entra ID** into **Google Cloud Identity**, and dataset-level *roles/bigquery.dataViewer* bindings are granted directly to those Cloud Identity groups. When a user signs in through **Gemini Enterprise** (or a custom frontend using **Workforce Identity Federation** token exchange), Google issues a short-lived OAuth 2.0 access token (*ya29...*) scoped to that human user.
> * **This Self-Contained Workshop Sandbox**: Because you are running inside an isolated project without a corporate Entra ID tenant or multiple browser logins, we represent the two employees using **IAM Service Accounts** (*persona-alex-exec* and *persona-elena-care*) and mint short-lived **OAuth 2.0 Bearer tokens** via *iamcredentials.googleapis.com*. Both personas receive project-level *roles/bigquery.jobUser* (which allows submitting a query job but grants zero table read permissions), while dataset-level *roles/bigquery.dataViewer* access is attached directly to each BigQuery dataset. Over the wire, inside BigQuery's IAM evaluator, and inside Cloud Audit Logs, the token mechanics and enforcement are identical.

Equivalent gcloud commands:
```bash
gcloud iam service-accounts create persona-alex-exec --display-name="Alex Novak (Admin - Full BI Access)" --project="$PROJECT_ID"
gcloud iam service-accounts create persona-elena-care --display-name="Elena Kovac (Specialist - CRM Only Access)" --project="$PROJECT_ID"
```

In [ ]:
# Step 0.6c: Pre-provision the 2 Workshop IAM Personas (Admin and Specialist) and Dataset-Level BigQuery ACLs
PERSONA_ADMIN_SA      = f"persona-alex-exec@{PROJECT_ID}.iam.gserviceaccount.com"
PERSONA_SPECIALIST_SA = f"persona-elena-care@{PROJECT_ID}.iam.gserviceaccount.com"
ACCOUNT_MEMBER        = f"serviceAccount:{ACCOUNT}" if ACCOUNT.endswith(".gserviceaccount.com") else f"user:{ACCOUNT}"

for sa_id, display in [
    ("persona-alex-exec",  "Alex Novak (Admin - Full BI Access)"),
    ("persona-elena-care", "Elena Kovac (Specialist - CRM Only Access)"),
]:
    email = f"{sa_id}@{PROJECT_ID}.iam.gserviceaccount.com"
    existing_sa = sh(f"gcloud iam service-accounts list --project={PROJECT_ID} --filter='email={email}' --format='value(email)'", quiet=True)
    if not existing_sa:
        sh(f"gcloud iam service-accounts create {sa_id} --display-name='{display}' --project={PROJECT_ID}")
    # A just-created service account can take a few seconds before IAM accepts it in a binding -> retry.
    for cmd in [
        # 1. Project-level roles/bigquery.jobUser (allows submitting SQL query jobs, grants ZERO table read access)
        f"gcloud projects add-iam-policy-binding {PROJECT_ID} --member='serviceAccount:{email}' --role='roles/bigquery.jobUser' --condition=None --quiet",
        # 2. Allow your account (user: or serviceAccount:) to mint short-lived OAuth 2.0 access tokens (ya29...) for this persona
        f"gcloud iam service-accounts add-iam-policy-binding {email} --member='{ACCOUNT_MEMBER}' --role='roles/iam.serviceAccountTokenCreator' --project={PROJECT_ID} --quiet",
    ]:
        for attempt in range(12):
            r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
            if r.returncode == 0:
                break
            time.sleep(10)
        else:
            print(r.stderr.strip()[-600:])
            raise RuntimeError(f"IAM binding failed for {email} - check that your account has Project IAM Admin and Service Account Admin")
    print(f"IAM persona ready: {email}")

# 3. Configure Dataset-Level BigQuery IAM (READER / roles/bigquery.dataViewer per dataset):
#    1. Admin (Alex) gets READER on all 3 datasets: telco_crm, telco_billing, telco_network
#    2. Specialist (Elena) gets READER ONLY on telco_crm (she is denied on telco_billing and telco_network)
DATASET_PERSONA_ACLS = {
    BQ_DATASET_CRM:     [PERSONA_ADMIN_SA, PERSONA_SPECIALIST_SA],
    BQ_DATASET_BILLING: [PERSONA_ADMIN_SA],
    BQ_DATASET_NETWORK: [PERSONA_ADMIN_SA],
}

for ds_id, allowed_principals in DATASET_PERSONA_ACLS.items():
    ds = bq.get_dataset(f"{PROJECT_ID}.{ds_id}")
    entries = [e for e in ds.access_entries if e.entity_id not in (PERSONA_ADMIN_SA, PERSONA_SPECIALIST_SA)]
    for principal in allowed_principals:
        entries.append(bigquery.AccessEntry(role="READER", entity_type="userByEmail", entity_id=principal))
    ds.access_entries = entries
    bq.update_dataset(ds, ["access_entries"])
    print(f"BigQuery dataset {ds_id:<15} READER ACL -> {[p.split('@')[0] for p in allowed_principals]}")

## Step 0.7: Install and Verify the Agents CLI (agents-cli)

The **Agents CLI** (*google-agents-cli==1.10.0*) provides a unified command-line workflow across the entire agent lifecycle, covering project scaffolding, dependency installation, local execution, interactive browser debugging, automated LLM-as-a-judge evaluation, managed Agent Runtime deployment, and Gemini Enterprise registration:

```bash
uv tool install --force --default-index https://pypi.org/simple "google-agents-cli==1.10.0"
uvx --from "google-agents-cli==1.10.0" google-agents-cli setup --skip-auth
agents-cli --version
agents-cli login --status
```

In [ ]:
# Step 0.7: Install the pinned Agents CLI (==1.10.0) and check that it is logged in
sh('uv tool install --force --default-index https://pypi.org/simple "google-agents-cli==1.10.0"', check=False)
sh('uvx --from "google-agents-cli==1.10.0" google-agents-cli setup --skip-auth', check=False)
print()
sh("agents-cli --version")
sh("agents-cli login --status", check=False)

## Step 0.8: Smoke Test gemini-3.8-flash on the eu Endpoint

**gemini-3.8-flash** is served from the *global*, *us*, and *eu* multi-region endpoints. Throughout this workshop we route model calls to the *eu* endpoint so every inference request is processed inside the European Union, while stateful runtime services (Agent Runtime, Sessions, Memory Bank, Sandbox, Agent Registry, Cloud Run) run in your regional compute plane (*europe-west4*).

> [!TIP]
> On a newly initialized project, IAM bindings can take 20 to 60 seconds to propagate after enabling *aiplatform.googleapis.com*. The cell below automatically retries until the *eu* endpoint responds.

In [ ]:
# Step 0.8: Verify model access with one Gemini call on the eu endpoint, retried while IAM propagates
import logging
from google import genai
logging.getLogger("google_genai.models").setLevel(logging.ERROR)  # suppress harmless AFC info log on one-shot generate_content

client = genai.Client(vertexai=True, project=PROJECT_ID, location=MODEL_LOCATION)

for attempt in range(12):
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents="In one sentence: what is Pulse Telco? (It is a fictional Central European 5G & fiber telecommunications provider.)",
        )
        print(resp.text)
        break
    except Exception as e:
        if "PERMISSION_DENIED" in str(e) or "403" in str(e):
            print(f"not ready yet (attempt {attempt+1}): permissions still propagating, retrying in 20s")
            time.sleep(20)
        else:
            raise
else:
    raise RuntimeError("Model call still denied after 4 minutes - check IAM and that aiplatform.googleapis.com is enabled")

## Step 0.9: Save Shared Configuration to workshop.env (Merge-Safe)

Every subsequent notebook (Lab 01 through Lab 11) loads *../workshop.env* in its opening code cell so your project identifiers, European location settings, model endpoints, BigQuery dataset names, and persona service accounts are shared automatically across the entire workshop.

The helper below **merges** these baseline keys into *workshop.env* rather than truncating the file, ensuring that re-running Lab 00 later never erases *PULSE_AGENT_ENGINE_ID* or other resource identifiers written by Labs 02 to 08.

In [ ]:
# Step 0.9: Save workshop.env (merge-safe: preserves any PULSE_* keys written by later labs)
ENV_FILE = REPO_ROOT / "workshop.env"

existing_kv = {}
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        if "=" in line and not line.strip().startswith("#"):
            k, v = line.split("=", 1)
            existing_kv[k.strip()] = v.strip()

existing_kv.update({
    "PROJECT_ID": PROJECT_ID,
    "PROJECT_NUMBER": PROJECT_NUMBER,
    "REGION": REGION,
    "MODEL_LOCATION": MODEL_LOCATION,
    "LIVE_LOCATION": LIVE_LOCATION,
    "BQ_LOCATION": BQ_LOCATION,
    "PULSE_SKILLS_LOCATION": PULSE_SKILLS_LOCATION,
    "MA_LOCATION": MA_LOCATION,
    "MODEL": MODEL,
    "AGENT_NAME": AGENT_NAME,
    "BQ_DATASET_CRM": BQ_DATASET_CRM,
    "BQ_DATASET_BILLING": BQ_DATASET_BILLING,
    "BQ_DATASET_NETWORK": BQ_DATASET_NETWORK,
    "PERSONA_ADMIN_SA": PERSONA_ADMIN_SA,
    "PERSONA_SPECIALIST_SA": PERSONA_SPECIALIST_SA,
    "GOOGLE_API_USE_CLIENT_CERTIFICATE": "false",
})

ENV_FILE.write_text("\n".join(f"{k}={v}" for k, v in existing_kv.items()) + "\n")
print(ENV_FILE.read_text())

### 🧪 Try it yourself (up to 3 min)

1. **Inspect a governed Knowledge Catalog entry (eu)**: Run the following command in a scratch cell or terminal to inspect the *telco-governance* and *telco-semantics* aspects attached to *telco_billing.invoices*:
   ```bash
   gcloud dataplex entries lookup --project="$PROJECT_ID" --location=eu \
     --entry="projects/$PROJECT_ID/locations/eu/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/$PROJECT_ID/datasets/telco_billing/tables/invoices"
   ```
   Observe that the required access tier is set to BILLING_AUTHORIZED and the canonical formula separates invoices 30 or more days past due (Delinquent) from disputed invoices.
2. **Verify dataset-level IAM ACLs**: Run the command below to inspect the dataset access control list on *telco_billing* and confirm that *persona-alex-exec* is listed as READER while *persona-elena-care* is absent:
   ```bash
   bq show --format=prettyjson "$PROJECT_ID:telco_billing" | jq '.access'
   ```

## 💶 Cost & scaling

| Service | How it is billed (list price, verified 2026-10-20) | This lab |
| --- | --- | --- |
| BigQuery EU: datasets + load | Batch load jobs are **free** (shared pool). Storage: **\$0.02/GiB-month** active, **\$0.01** long-term; **first 10 GiB/month free**. Queries: **\$6.25/TiB** on-demand, **first 1 TiB/month free** | ≈ **\$0.00** (≈ 200 KB of data) |
| Knowledge Catalog eu: aspect types | API calls and catalog search are **free**. Metadata you create (aspect types, aspects, entries): **\$2.00/GiB-month** | ≈ **\$0.00** (KB of metadata) |
| IAM, service accounts | **No charge** | \$0.00 |
| Gemini 3.8 Flash eu smoke test | **\$1.65** per 1M input tokens and **\$8.25** per 1M output tokens (eu non-global endpoint, +10% vs global). Through 2026-12-31, an introductory 50% credit applies on net spend | < \$0.001 |

**How it scales.** BigQuery on-demand provides up to 2,000 concurrent slots per project, with a 10 MB minimum billed per referenced table per query. Gemini Standard PayGo has no fixed per-project quota; the organization's prioritized throughput threshold scales with 30-day spend (Flash Tier 1 = 2M tokens/min; *eu* operates from its own regional capacity pool).

**What this lab cost you: ≈ \$0.00.**

*Pricing references:* [BigQuery pricing](https://cloud.google.com/bigquery/pricing) · [Knowledge Catalog pricing](https://cloud.google.com/products/knowledge-catalog/pricing) · [Gemini platform generative AI pricing](https://cloud.google.com/gemini-enterprise-agent-platform/generative-ai/pricing) · [IAM pricing](https://cloud.google.com/iam/pricing)

## ✅ Lab 00 Complete: Recap

* **Project & APIs**: Your workshop project has all ~30 GEAP, BigQuery, Knowledge Catalog, Agent Registry, Agent Gateway, Model Armor, and IAM Credentials APIs enabled.
* **Locations Configured**: **europe-west4** for regional stateful runtime services; **eu** multi-region for Gemini 3.8 Flash, Gemini 3.8 Live, Skill Registry, Model Armor, and Knowledge Catalog; **EU** multi-region for BigQuery.
* **Multi-Domain BigQuery (EU)**: The *telco_crm*, *telco_billing*, and *telco_network* datasets are seeded with six tables and stamped with governance metadata.
* **Two IAM User Personas & Dataset-Level BigQuery ACLs**:
  * **Admin (*persona-alex-exec*)**: Granted read access across all three BigQuery datasets (CRM, Billing, and Network).
  * **Specialist (*persona-elena-care*)**: Granted read access **only** on the CRM dataset (denied on Billing and Network by BigQuery IAM).
* **Knowledge Catalog (eu)**: Custom Aspect Types *telco-governance* and *telco-semantics* are created and attached to the BigQuery table entries.
* **CLI & Model Connectivity**: The Agents CLI (1.10.0) is installed and **gemini-3.8-flash** is responding on the *eu* endpoint.
* **State Persisted**: Shared configuration is written (merge-safe) to *workshop.env* in the repository root.

👉 **Next:** Proceed to **[Lab 01: Build the Multi-Agent Pulse Concierge Locally with ADK](lab01_local_agent.ipynb)**.